# Lesson 7: Building Autonomous Agents with LangGraph

Welcome to Lesson 7! In every lesson so far, **we** decided the execution path. We told the chain exactly which steps to run and in what order. Today, we hand that control over to the LLM itself.

### The Session Goal
We will build a fully autonomous **ReAct Agent** using **LangGraph** -- LangChain's framework for cyclic, stateful AI workflows. The agent will reason about a problem, decide which tool to use, observe the result, and repeat until it has a final answer.

### The Core Concepts
1. **ReAct Pattern**: The Reason + Act loop that gives LLMs decision-making ability.
2. **LangGraph State Machine**: Nodes (actions), edges (transitions), and conditional routing.
3. **Tool Binding**: Giving the agent callable functions it can invoke autonomously.
4. **The Agent Loop**: Observe -> Think -> Act -> repeat until the task is complete.

### How This Connects to Previous Lessons
| Lesson | Concept | Role in Today's Agent |
|--------|---------|----------------------|
| 2 | Tools | The agent's available actions |
| 3 | Structured Output | How the agent formats tool calls |
| 4 | Memory | The agent's state persists across turns |
| 5 | LCEL Chains | Internal wiring of each node |
| 6 | RAG Retrieval | Can be one of the agent's tools |

In [ ]:
!pip install -q langchain-core langchain-openai langgraph

import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## Step 1: The Problem -- Why Chains Are Not Enough

In Lesson 5, we built sequential chains: Step A always leads to Step B, which always leads to Step C. This is **deterministic** and **linear**.

But real-world tasks are not linear:
- "What is the weather in the city where the next Olympics will be held?" requires multiple lookups.
- "Calculate 15% tip on my $84.50 dinner bill and convert it to euros" requires a calculator AND a currency lookup.
- The model cannot know in advance how many steps it needs or which tools to call.

### The ReAct Pattern (Reason + Act)
```text
Loop:
  1. THINK  -- The LLM reasons about what to do next
  2. ACT    -- The LLM calls a tool with specific arguments
  3. OBSERVE -- The tool result is fed back to the LLM
  4. REPEAT -- Until the LLM decides it has a final answer
```

This is fundamentally different from a chain. The LLM is now the **decision engine**, not just a text generator.

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Ask a question that requires real-time computation
response = model.invoke("What is 4782 * 9.7 + 312.5?")
print("--- Raw LLM Attempting Math ---")
print(f"AI Response: {response.content}")
print(f"\nActual Answer: {4782 * 9.7 + 312.5}")
print("\nThe LLM may get this wrong or approximate. It cannot reliably execute calculations.")

---

## Step 2: Defining Tools for the Agent

Before building the agent, we need to define the **tools** it can use. A tool is simply a Python function decorated with `@tool` that includes a clear docstring (the LLM reads this to decide when to use it).

We will create three tools:
1. **multiply** -- Multiplies two numbers
2. **add** -- Adds two numbers
3. **web_lookup** -- Simulates looking up real-time information

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together. Use this when you need to calculate a product."""
    return a * b

@tool
def add(a: float, b: float) -> float:
    """Add two numbers together. Use this when you need to calculate a sum."""
    return a + b

@tool
def web_lookup(query: str) -> str:
    """Look up real-time information from the web. Use this for current events, prices, or live data."""
    # Simulated responses for demonstration
    data = {
        "euro exchange rate": "1 USD = 0.92 EUR (as of today)",
        "weather paris": "Paris: 18C, partly cloudy",
        "2028 olympics city": "The 2028 Summer Olympics will be held in Los Angeles, USA",
    }
    for key, value in data.items():
        if key in query.lower():
            return value
    return f"No results found for: {query}"

# Collect tools into a list
tools = [multiply, add, web_lookup]

print("--- Defined Tools ---")
for t in tools:
    print(f"  {t.name}: {t.description}")

---

## Step 3: Binding Tools to the Model

LangChain's `.bind_tools()` method tells the LLM what tools are available. The model does NOT execute the tools itself -- it outputs a structured **tool call request** specifying which tool to use and with what arguments. Our code then executes the tool and feeds the result back.

This is the key insight: the LLM is a **decision maker**, not an executor.

In [ ]:
from langchain_openai import ChatOpenAI

# Bind tools to the model -- this makes the model aware of available tools
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
model_with_tools = model.bind_tools(tools)

# Ask a question that requires a tool
response = model_with_tools.invoke("What is 4782 multiplied by 9.7?")

print("--- Model Response with Tool Binding ---")
print(f"Content: '{response.content}'")
print(f"Tool Calls: {response.tool_calls}")
print("\nNotice: The model did NOT answer directly. It requested a tool call with specific arguments.")
print("Our code must now execute this tool and feed the result back.")

---

## Step 4: Building the Agent with LangGraph

LangGraph models agent workflows as a **state graph**:
- **State**: A dictionary that accumulates messages (the conversation so far)
- **Nodes**: Functions that process state (e.g., "call the LLM", "execute tools")
- **Edges**: Define the flow between nodes (including conditional routing)

### The Agent Graph Structure:
```text
        START
          |
          v
    [agent_node]  <----+
          |            |
    (has tool calls?)  |
      /        \       |
    YES         NO     |
     |           |     |
     v           v     |
[tool_node]    END     |
     |                 |
     +-----------------+
```

The **loop** is what makes this an agent, not a chain. The LLM keeps calling tools until it decides it has the final answer.

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

# 1. Initialize the model
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. Create a ReAct agent with LangGraph's prebuilt helper
# This constructs the full graph: agent node -> conditional edge -> tool node -> loop back
agent = create_react_agent(model, tools)

print("--- ReAct Agent Created ---")
print(f"Tools available: {[t.name for t in tools]}")
print("The agent can now autonomously decide which tools to call and when to stop.")

---

## Step 5: Running the Agent -- Single Tool Call

Let's start simple. We ask a question that requires exactly one tool call. Watch how the agent reasons, calls the tool, observes the result, and then provides a final answer.

In [ ]:
from langchain_core.messages import HumanMessage

# Invoke the agent with a simple math question
result = agent.invoke(
    {"messages": [HumanMessage(content="What is 4782 multiplied by 9.7?")]}
)

print("--- Agent Execution Trace ---\n")
for msg in result["messages"]:
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print(f"[{role}] Requesting tool: {msg.tool_calls[0]['name']}({msg.tool_calls[0]['args']})")
    elif hasattr(msg, "name") and msg.name:
        print(f"[ToolMessage] {msg.name} returned: {msg.content}")
    else:
        print(f"[{role}] {msg.content}")
    print()

---

## Step 6: Multi-Step Reasoning -- The Agent Loop in Action

Now let's ask a question that requires **multiple tool calls in sequence**. The agent must:
1. Look up information (web_lookup)
2. Perform a calculation (multiply)
3. Perform another calculation (add)

Watch how the agent autonomously chains these steps without us writing any routing logic.

In [ ]:
# A multi-step question requiring multiple tool calls
question = "I have a dinner bill of $84.50. Calculate a 15% tip, then add it to the bill for the total."

result = agent.invoke(
    {"messages": [HumanMessage(content=question)]}
)

print("--- Multi-Step Agent Trace ---\n")
print(f"Question: {question}\n")
step = 1
for msg in result["messages"][1:]:  # Skip the initial human message
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  Step {step} [THINK+ACT]: Call {tc['name']}({tc['args']})")
            step += 1
    elif hasattr(msg, "name") and msg.name:
        print(f"         [OBSERVE]: {msg.name} -> {msg.content}")
    else:
        print(f"\n  FINAL ANSWER: {msg.content}")

---

## Step 7: Adding a System Prompt -- Controlling Agent Behavior

We can steer the agent's personality and constraints using a system prompt. This is essential in production to:
- Restrict what the agent is allowed to do
- Set a consistent tone
- Provide domain-specific instructions

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

# Create an agent with a system prompt that constrains its behavior
system_prompt = """You are a helpful financial assistant. 
You MUST use the available tools for any calculations -- never do math in your head.
Always show your work by explaining each step clearly.
If you cannot answer a question with your available tools, say so honestly."""

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
financial_agent = create_react_agent(model, tools, prompt=system_prompt)

# Test it
result = financial_agent.invoke(
    {"messages": [HumanMessage(content="What is 250 multiplied by 12, then add 99.50 to that?")]}
)

print("--- Financial Agent with System Prompt ---\n")
for msg in result["messages"][1:]:
    role = msg.__class__.__name__
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  [TOOL CALL] {tc['name']}({tc['args']})")
    elif hasattr(msg, "name") and msg.name:
        print(f"  [RESULT]    {msg.name} -> {msg.content}")
    else:
        print(f"\n  [FINAL]: {msg.content}")

---

## Step 8: Building the Graph Manually (Under the Hood)

The `create_react_agent` helper is convenient, but understanding what it builds internally is crucial for customization. Let's construct the same agent graph manually using LangGraph primitives.

This reveals the actual architecture: **StateGraph** with typed state, explicit nodes, and conditional edges.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI

# 1. Define the model with tools bound
model = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)

# 2. Define the agent node -- calls the LLM
def agent_node(state: MessagesState):
    """The 'thinking' node: the LLM decides what to do next."""
    response = model.invoke(state["messages"])
    return {"messages": [response]}

# 3. Define the conditional edge -- routes based on whether tool calls exist
def should_continue(state: MessagesState):
    """If the last message has tool calls, go to tools. Otherwise, end."""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

# 4. Build the graph
workflow = StateGraph(MessagesState)

# Add nodes
workflow.add_node("agent", agent_node)
workflow.add_node("tools", ToolNode(tools))

# Add edges
workflow.add_edge(START, "agent")           # Start -> Agent
workflow.add_conditional_edges("agent", should_continue, ["tools", END])  # Agent -> Tools or END
workflow.add_edge("tools", "agent")         # Tools -> Agent (the loop!)

# Compile into a runnable
custom_agent = workflow.compile()

print("--- Custom Agent Graph Built ---")
print("Nodes: agent, tools")
print("Flow:  START -> agent -> (tools -> agent)* -> END")
print("\nThis is identical to create_react_agent but now fully customizable.")

In [ ]:
# Test our custom-built agent
result = custom_agent.invoke(
    {"messages": [HumanMessage(content="Where will the 2028 Olympics be held? Then multiply 2028 by 3.")]}
)

print("--- Custom Agent: Multi-Tool Execution ---\n")
for msg in result["messages"]:
    role = msg.__class__.__name__
    if role == "HumanMessage":
        print(f"  [USER] {msg.content}\n")
    elif hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  [THINK+ACT] Call {tc['name']}({tc['args']})")
    elif hasattr(msg, "name") and msg.name:
        print(f"  [OBSERVE]   {msg.name} returned: {msg.content}")
    else:
        print(f"\n  [FINAL ANSWER] {msg.content}")

---

## Step 9: Streaming Agent Steps in Real Time

In production, you want to show users what the agent is doing as it works (not just the final answer). LangGraph supports streaming each step as it happens.

In [ ]:
# Stream each step of the agent's execution
print("--- Streaming Agent Steps ---\n")

question = "Look up the euro exchange rate, then multiply 84.50 by that rate."

for step in custom_agent.stream(
    {"messages": [HumanMessage(content=question)]},
    stream_mode="updates"
):
    for node_name, output in step.items():
        print(f">> Node '{node_name}' executed:")
        last_msg = output["messages"][-1]
        if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
            for tc in last_msg.tool_calls:
                print(f"   Tool request: {tc['name']}({tc['args']})")
        elif hasattr(last_msg, "name") and last_msg.name:
            print(f"   Tool result: {last_msg.content}")
        else:
            print(f"   Final: {last_msg.content}")
        print()

---

## Step 10: Deterministic Routing vs. Agent Routing

Not every routing decision needs an agent loop. Sometimes you know in advance that "math questions go to chain A" and "history questions go to chain B." This is **deterministic routing** using `RunnableBranch`.

### The Trade-off:

| Approach | When to Use | Cost | Flexibility |
|----------|-------------|------|-------------|
| `RunnableBranch` | Categories are known, routes are fixed | 1 LLM call to classify + 1 to answer | Low (new categories require code changes) |
| Agent (ReAct) | Tasks are open-ended, tools may combine | N LLM calls (one per loop iteration) | High (agent picks tools dynamically) |

Let's build a deterministic router and then show how the same problem is handled by our agent.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# 1. Define specialized chains for each category
math_prompt = ChatPromptTemplate.from_template(
    "You are a math tutor. Solve this step by step:\n{input}"
)
math_chain = math_prompt | model | parser

history_prompt = ChatPromptTemplate.from_template(
    "You are a history expert. Answer this concisely:\n{input}"
)
history_chain = history_prompt | model | parser

general_prompt = ChatPromptTemplate.from_template(
    "Answer this question helpfully:\n{input}"
)
general_chain = general_prompt | model | parser

# 2. Classification chain -- determines the route
classifier_prompt = ChatPromptTemplate.from_template(
    "Classify this question into exactly one category: 'math', 'history', or 'general'.\n"
    "Question: {input}\n"
    "Category:"
)
classifier_chain = classifier_prompt | model | parser

# 3. Build the deterministic router with RunnableBranch
def classify_and_route(input_dict):
    category = classifier_chain.invoke(input_dict).strip().lower()
    return category

router = (
    RunnableLambda(lambda x: {**x, "category": classify_and_route(x)})
    | RunnableBranch(
        (lambda x: "math" in x["category"], lambda x: math_chain.invoke(x)),
        (lambda x: "history" in x["category"], lambda x: history_chain.invoke(x)),
        lambda x: general_chain.invoke(x),  # Default fallback
    )
)

# 4. Test the deterministic router
print("--- Deterministic Router (RunnableBranch) ---\n")
test_questions = [
    "What is 145 * 32 + 18?",
    "When did World War II end?",
    "What is a good recipe for pasta?",
]

for q in test_questions:
    answer = router.invoke({"input": q})
    print(f"Q: {q}")
    print(f"A: {answer[:100]}...")
    print()

In [ ]:
# 5. Now contrast: the agent handles the same questions WITHOUT explicit routing
# It dynamically picks the right tool based on the question content

print("--- Agent-Based Routing (No Explicit Router Needed) ---\n")

# The same math question, routed automatically by the agent's reasoning
result = custom_agent.invoke(
    {"messages": [HumanMessage(content="What is 145 * 32 + 18?")]}
)

final_msg = result["messages"][-1]
print(f"Q: What is 145 * 32 + 18?")
print(f"A: {final_msg.content}")
print()
print("Key difference:")
print("  Router: We wrote explicit classification logic (2 LLM calls: classify + answer)")
print("  Agent:  The LLM decided on its own which tools to use (N calls until done)")
print()
print("Use routers when categories are stable and cost matters.")
print("Use agents when tasks are unpredictable or require multi-tool composition.")

---

## Summary and Key Takeaways

Today we crossed the threshold from **deterministic chains** to **autonomous agents**. Here is what changed:

| Concept | Chain (Lesson 5) | Router (Step 10) | Agent (Steps 4-9) |
|---------|-------------------|------------------|---------------------|
| Flow | Linear, hardcoded | Branching, hardcoded | Cyclic, dynamic |
| Decision maker | The developer | Classifier LLM + developer | The LLM fully |
| Steps | Fixed at design time | Fixed per category | Determined at runtime |
| Cost | 1 LLM call | 2 LLM calls | N LLM calls |
| Flexibility | None | Add new branches in code | Fully autonomous |

### What We Built
1. **Tool definitions** with `@tool` decorator and clear docstrings
2. **Tool binding** via `.bind_tools()` so the LLM knows its options
3. **Prebuilt ReAct agent** with `create_react_agent` for quick setup
4. **Custom StateGraph** with explicit nodes, edges, and conditional routing
5. **Streaming execution** for real-time visibility into agent reasoning
6. **Deterministic router** with `RunnableBranch` vs. agent-based routing contrast

### When to Use What
- **LCEL Chain**: The path is always the same (e.g., translate -> summarize)
- **RunnableBranch Router**: Categories are known and stable, cost matters, no tool composition needed
- **ReAct Agent**: Tasks are open-ended, may require multiple tools, or the number of steps is unknown

### Production Considerations
- **Max iterations**: Always set a recursion limit to prevent infinite loops
- **Human-in-the-loop**: Add checkpoints where a human must approve before the agent acts
- **Error handling**: Wrap tool execution in try/except to give the agent a chance to recover
- **Observability**: Use LangSmith tracing to debug agent decisions in production
- **Cost control**: Each loop iteration costs an LLM call; monitor token usage

### The Full Course Arc
You now have every building block of a production AI system:
1. Raw LLM capabilities and limitations
2. Tools that extend what an LLM can do
3. Structured prompts for reliable output
4. Memory for multi-turn conversations
5. LCEL chains for deterministic pipelines
6. RAG for grounding answers in real documents
7. Agents and routers that orchestrate all of the above